# Merge Physics Results with Usage

- `statement_row_id` → `N`
- `№` → answer-key `N`
- `Ans number` → `actual_numeric`
- `Ans formula` → `actual_formula`
- `units` → `actual_units`
- `usage.prompt_tokens` → `input_tokens`
- `usage.completion_tokens` → `output_tokens`
- `usage.total_tokens` → `total_tokens`
- `usage.cost` → `cost`


In [ ]:
import pandas as pd
import numpy as np
import re
import json
import ast
from pathlib import Path

try:
    from google.colab import files
except ImportError:
    files = None


In [ ]:
RESULTS_DIR = Path("results")
COMBINED_CSV = "combined_csv_files.csv"
ANSWERS_CSV = "Physics Problems 2026 march 7 - Sheet1.csv"
OUTPUT_CSV = "physics_results_with_usage.csv"

if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f"Could not find the results folder: {RESULTS_DIR.resolve()}\n"
        "Create a folder named 'results' and put the individual result CSV files inside it."
    )

csv_paths = sorted(
    p for p in RESULTS_DIR.glob("*.csv")
    if p.name not in {COMBINED_CSV, ANSWERS_CSV, OUTPUT_CSV}
)

if not csv_paths:
    raise FileNotFoundError(
        f"No CSV files found in {RESULTS_DIR.resolve()}\n"
        "Put the individual result CSV files inside the 'results' folder."
    )

frames = []
for csv_path in csv_paths:
    df = pd.read_csv(csv_path)

    # Keep the model/source traceable after concatenation.
    # If the CSV has no ai_model/model/source_file column, source_file will be used later.
    lower_cols = {str(c).lower().strip() for c in df.columns}
    if not {"ai_model", "model", "source_file"}.intersection(lower_cols):
        df["source_file"] = csv_path.stem

    frames.append(df)

results = pd.concat(frames, ignore_index=True, sort=False)
results.to_csv(COMBINED_CSV, index=False)

if not Path(ANSWERS_CSV).exists():
    if files is None:
        raise FileNotFoundError(
            f"Could not find {ANSWERS_CSV}. Put it next to this notebook/script and rerun."
        )
    print(f"Upload the answer-key CSV now: {ANSWERS_CSV}")
    files.upload()

if not Path(ANSWERS_CSV).exists():
    raise FileNotFoundError(
        f"Still could not find {ANSWERS_CSV}. "
        "Check that the uploaded answer-key filename matches exactly."
    )

answers = pd.read_csv(ANSWERS_CSV)

print(f"Merged {len(csv_paths)} CSV files from {RESULTS_DIR}/ into {COMBINED_CSV}")
print("Merged files:")
for csv_path in csv_paths:
    print(f"- {csv_path}")

print("\nCombined CSV columns:")
print(list(results.columns))
print("\nPhysics answer CSV columns:")
print(list(answers.columns))

print("\nRows in combined CSV:", len(results))
print("Rows in answer CSV:", len(answers))


Merged 23 CSV files from results/ into combined_csv_files.csv
Merged files:
- results/anthropicclaude-opus-4.7.csv
- results/anthropicclaude-sonnet-4.5.csv
- results/deepseekdeepseek-r1.csv
- results/deepseekdeepseek-v3.2-speciale.csv
- results/googlegemini-2.5-pro.csv
- results/googlegemini-3.1-pro-preview.csv
- results/googlegemma-4-31b-it.csv
- results/meta-llamallama-4-maverick.csv
- results/microsoftphi-4.csv
- results/mistralaiministral-14b-2512.csv
- results/mistralaipixtral-large-2411.csv
- results/moonshotaikimi-k2.5.csv
- results/moonshotaikimi-k2.6.csv
- results/nvidiallama-3.3-nemotron-super-49b-v1.5.csv
- results/openaigpt-5.4-pro.csv
- results/openaigpt-5.4.csv
- results/openaigpt-oss-120b.csv
- results/openaio4-mini-high.csv
- results/qwenqwen3.5-397b-a17b.csv
- results/qwenqwen3.6-plus.csv
- results/stepfunstep-3.5-flash.csv
- results/x-aigrok-4.20.csv
- results/xiaomimimo-v2.5-pro.csv

Combined CSV columns:
['datetime', 'ai_model', 'statement_row_id', 'statement', 'ok'

In [ ]:
def normalize_colnames(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def find_col(df, candidates, label):
    """Find a column using exact case-insensitive aliases."""
    lower_map = {str(c).lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = str(cand).lower().strip()
        if key in lower_map:
            return lower_map[key]
    raise KeyError(
        f"Could not find column for {label}. Tried: {candidates}. "
        f"Available columns: {list(df.columns)}"
    )

def find_col_optional(df, candidates):
    lower_map = {str(c).lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = str(cand).lower().strip()
        if key in lower_map:
            return lower_map[key]
    return None

def numeric_to_2dp(value):
    """Extract the first numeric value and round to 2 decimal places."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text == '' or text.lower() in {'nan', 'null', 'none'}:
        return np.nan
    match = re.search(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', text)
    if not match:
        return np.nan
    try:
        return round(float(match.group(0)), 2)
    except ValueError:
        return np.nan

def clean_formula(value):
    """Remove leading assignments like x = ..., v_final = ..., answer = ... and normalize spaces."""
    if pd.isna(value):
        return ''
    text = str(value).strip()
    if text.lower() in {'nan', 'null', 'none'}:
        return ''
    text = re.sub(r'^\s*[A-Za-z][A-Za-z0-9_]*\s*=\s*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_full_api_response(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    text = str(value).strip()
    if text == '' or text.lower() in {'nan', 'null', 'none'}:
        return {}
    for parser in (ast.literal_eval, json.loads):
        try:
            parsed = parser(text)
            if isinstance(parsed, dict):
                return parsed
            if isinstance(parsed, str):
                parsed_again = parser(parsed)
                if isinstance(parsed_again, dict):
                    return parsed_again
        except Exception:
            pass
    return {}

def first_present(mapping, keys):
    for key in keys:
        if isinstance(mapping, dict) and key in mapping and mapping[key] is not None:
            return mapping[key]
    return np.nan

def extract_usage_from_response(value):
    response = parse_full_api_response(value)
    usage = response.get('usage', {}) if isinstance(response, dict) else {}
    if not isinstance(usage, dict):
        usage = {}

    input_tokens = first_present(usage, ['input_tokens', 'prompt_tokens'])
    output_tokens = first_present(usage, ['output_tokens', 'completion_tokens'])
    total_tokens = first_present(usage, ['total_tokens'])
    cost = first_present(usage, ['cost', 'total_cost', 'estimated_cost'])

    cost_details = usage.get('cost_details', {}) if isinstance(usage, dict) else {}
    if pd.isna(cost) and isinstance(cost_details, dict):
        cost = first_present(cost_details, ['upstream_inference_cost', 'total_cost', 'cost'])

    return {
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': total_tokens,
        'cost': cost,
    }

def usage_columns_from_results(results, full_api_response_col):
    usage_df = pd.DataFrame(
        list(results[full_api_response_col].apply(extract_usage_from_response)),
        index=results.index,
    )

    fallback_map = {
        'input_tokens': ['input_tokens', 'prompt_tokens'],
        'output_tokens': ['output_tokens', 'completion_tokens'],
        'total_tokens': ['total_tokens'],
        'cost': ['cost'],
    }

    for target_col, candidates in fallback_map.items():
        for fallback_col in candidates:
            source_col = find_col_optional(results, [fallback_col])
            if source_col is not None:
                usage_df[target_col] = usage_df[target_col].fillna(
                    pd.to_numeric(results[source_col], errors='coerce')
                )
                break

    usage_df['input_tokens'] = pd.to_numeric(usage_df['input_tokens'], errors='coerce')
    usage_df['output_tokens'] = pd.to_numeric(usage_df['output_tokens'], errors='coerce')
    usage_df['total_tokens'] = pd.to_numeric(usage_df['total_tokens'], errors='coerce')
    usage_df['cost'] = pd.to_numeric(usage_df['cost'], errors='coerce')

    missing_total = usage_df['total_tokens'].isna() & usage_df['input_tokens'].notna() & usage_df['output_tokens'].notna()
    usage_df.loc[missing_total, 'total_tokens'] = usage_df.loc[missing_total, 'input_tokens'] + usage_df.loc[missing_total, 'output_tokens']

    for token_col in ['input_tokens', 'output_tokens', 'total_tokens']:
        usage_df[token_col] = usage_df[token_col].round().astype('Int64')

    return usage_df[['input_tokens', 'output_tokens', 'total_tokens', 'cost']]


In [ ]:
results = normalize_colnames(results)
answers = normalize_colnames(answers)

# Result file aliases
res_n = find_col(results, ['N', 'statement_row_id', 'statement id', 'number', 'problem', 'problem_number'], 'result problem number')
res_datetime = find_col(results, ['datetime', 'date_time', 'timestamp'], 'result datetime')
res_ai_model = find_col(results, ['ai_model', 'model', 'source_file'], 'result AI model')
res_final_numeric = find_col(results, ['final_numeric'], 'result final numeric')
res_final_formula = find_col(results, ['final_formula'], 'result final formula')
res_final_units = find_col(results, ['final_units', 'final_unit'], 'result final units')
res_full_api_response = find_col(results, ['full_api_response', 'api_response', 'response'], 'result full API response')

# Answer-key aliases
ans_n = find_col(answers, ['N', '№', 'No', 'number', 'problem', 'problem_number', 'statement_row_id'], 'answer-key problem number')
ans_actual_numeric = find_col(answers, ['actual_numeric', 'Ans number', 'Ans Number', 'answer_numeric', 'final_numeric', 'numeric'], 'answer-key numeric answer')
ans_actual_formula = find_col(answers, ['actual_formula', 'Ans LaTeX', 'answer_formula', 'final_formula', 'formula'], 'answer-key formula answer')
ans_actual_units = find_col(answers, ['actual_units', 'units', 'Units', 'answer_units', 'final_units', 'final_unit'], 'answer-key units answer')

# Optional formula fallback for rows where `Ans formula` is blank but `Ans LaTeX` exists.
answer_lower_cols = {str(c).lower().strip(): c for c in answers.columns}
ans_latex_col = answer_lower_cols.get('ans latex')

print('Column mapping:')
print({
    'result_N': res_n,
    'result_datetime': res_datetime,
    'result_ai_model': res_ai_model,
    'result_final_numeric': res_final_numeric,
    'result_final_formula': res_final_formula,
    'result_final_units': res_final_units,
    'result_full_api_response': res_full_api_response,
    'answer_N': ans_n,
    'answer_actual_numeric': ans_actual_numeric,
    'answer_actual_formula': ans_actual_formula,
    'answer_actual_units': ans_actual_units,
    'answer_formula_fallback': ans_latex_col,
})

usage_df = usage_columns_from_results(results, res_full_api_response)

results_clean = results[[res_n, res_datetime, res_ai_model, res_final_numeric, res_final_formula, res_final_units]].copy()
results_clean.columns = ['N', 'datetime', 'ai_model', 'final_numeric', 'final_formula', 'final_units']
for col in ['input_tokens', 'output_tokens', 'total_tokens', 'cost']:
    results_clean[col] = usage_df[col].values

answers_clean = answers[[ans_n, ans_actual_numeric, ans_actual_formula, ans_actual_units]].copy()
answers_clean.columns = ['N', 'actual_numeric', 'actual_formula', 'actual_units']

if ans_latex_col is not None:
    blank_formula = answers_clean['actual_formula'].isna() | (answers_clean['actual_formula'].astype(str).str.strip() == '')
    answers_clean.loc[blank_formula, 'actual_formula'] = answers.loc[blank_formula, ans_latex_col]

results_clean['N'] = pd.to_numeric(results_clean['N'], errors='coerce').astype('Int64')
answers_clean['N'] = pd.to_numeric(answers_clean['N'], errors='coerce').astype('Int64')

merged = results_clean.merge(answers_clean, on='N', how='left', indicator=True)

unmatched = merged['_merge'].ne('both')
if unmatched.any():
    print('Warning: rows with no matching answer-key problem number:', int(unmatched.sum()))

missing_usage = merged[['input_tokens', 'output_tokens', 'total_tokens', 'cost']].isna().any(axis=1)
if missing_usage.any():
    print('Warning: rows with at least one missing usage/cost value:', int(missing_usage.sum()))

merged = merged.drop(columns=['_merge'])
display(merged.head())


Column mapping:
{'result_N': 'statement_row_id', 'result_datetime': 'datetime', 'result_ai_model': 'ai_model', 'result_final_numeric': 'final_numeric', 'result_final_formula': 'final_formula', 'result_final_units': 'final_units', 'result_full_api_response': 'full_api_response', 'answer_N': '№', 'answer_actual_numeric': 'Ans number', 'answer_actual_formula': 'Ans LaTeX', 'answer_actual_units': 'units', 'answer_formula_fallback': 'Ans LaTeX'}


,N,datetime,ai_model,final_numeric,final_formula,final_units,input_tokens,output_tokens,total_tokens,cost,actual_numeric,actual_formula,actual_units
0,1,2026-04-23-15-19-16,anthropic/claude-opus-4.7,1000.00000,v = \sqrt{l a},m/s,822,105,927,0.006735,1000.00,\sqrt{a l},m/s
1,2,2026-04-23-15-19-21,anthropic/claude-opus-4.7,16.00000,x = At + Bt^3 \text{ at } t=\sqrt{-A/(3B)},m,859,360,1219,0.013295,-0.50,A + B \left(t_{1}^{2} + t_{1} t_{2} + t_{2}^{2...,m/s
2,3,2026-04-23-15-19-26,anthropic/claude-opus-4.7,0.28764,t = \frac{32 \cdot 80}{80^2 + 50^2} = \frac{12...,h,809,223,1032,0.009620,0.29,\frac{d v_{1}}{v_{1}^{2} + v_{2}^{2}},h
3,4,2026-04-23-15-19-31,anthropic/claude-opus-4.7,4.47000,a = 2\sqrt{\gamma_1^2 + \gamma_2^2} = 2\sqrt{5...,m/s^2,898,429,1327,0.015215,-,\arccos{\left(\frac{ax vx + ay vy}{a v} \right)},degrees
4,5,2026-04-23-15-19-34,anthropic/claude-opus-4.7,160.00000,\frac{\frac{1}{2}mv^2}{t},W,847,53,900,0.005560,160.00,\frac{m \left(v_{0}^{2} - vf^{2}\right)}{2 \De...,J/s


In [ ]:
merged['final_numeric_2dp'] = merged['final_numeric'].apply(numeric_to_2dp)
merged['actual_numeric_2dp'] = merged['actual_numeric'].apply(numeric_to_2dp)

merged['final_formula_clean'] = merged['final_formula'].apply(clean_formula)
merged['actual_formula_clean'] = merged['actual_formula'].apply(clean_formula)

# Output values requested by user:
# - numeric columns rounded to 2 decimals
# - formula columns cleaned of leading assignments like x = ...
# - score columns removed
# - token/cost columns extracted from full_api_response usage data
final_df = pd.DataFrame({
    'N': merged['N'],
    'datetime': merged['datetime'],
    'ai_model': merged['ai_model'],
    'final_numeric': merged['final_numeric_2dp'],
    'final_formula': merged['final_formula_clean'],
    'final_units': merged['final_units'],
    'actual_numeric': merged['actual_numeric_2dp'],
    'actual_formula': merged['actual_formula_clean'],
    'actual_units': merged['actual_units'],
    'input_tokens': merged['input_tokens'],
    'output_tokens': merged['output_tokens'],
    'total_tokens': merged['total_tokens'],
    'cost': merged['cost'],
})

display(final_df.head())


,N,datetime,ai_model,final_numeric,final_formula,final_units,actual_numeric,actual_formula,actual_units,input_tokens,output_tokens,total_tokens,cost
0,1,2026-04-23-15-19-16,anthropic/claude-opus-4.7,1000.00,\sqrt{l a},m/s,1000.00,\sqrt{a l},m/s,822,105,927,0.006735
1,2,2026-04-23-15-19-21,anthropic/claude-opus-4.7,16.00,At + Bt^3 \text{ at } t=\sqrt{-A/(3B)},m,-0.50,A + B \left(t_{1}^{2} + t_{1} t_{2} + t_{2}^{2...,m/s,859,360,1219,0.013295
2,3,2026-04-23-15-19-26,anthropic/claude-opus-4.7,0.29,\frac{32 \cdot 80}{80^2 + 50^2} = \frac{128}{445},h,0.29,\frac{d v_{1}}{v_{1}^{2} + v_{2}^{2}},h,809,223,1032,0.009620
3,4,2026-04-23-15-19-31,anthropic/claude-opus-4.7,4.47,"2\sqrt{\gamma_1^2 + \gamma_2^2} = 2\sqrt{5}\,\...",m/s^2,NaN,\arccos{\left(\frac{ax vx + ay vy}{a v} \right)},degrees,898,429,1327,0.015215
4,5,2026-04-23-15-19-34,anthropic/claude-opus-4.7,160.00,\frac{\frac{1}{2}mv^2}{t},W,160.00,\frac{m \left(v_{0}^{2} - vf^{2}\right)}{2 \De...,J/s,847,53,900,0.005560


In [ ]:
final_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}')

print('\nUsage totals:')
print(final_df[['input_tokens', 'output_tokens', 'total_tokens', 'cost']].sum(numeric_only=True))

if files is not None:
    files.download(OUTPUT_CSV)
else:
    print(f'Output is available at: {Path(OUTPUT_CSV).resolve()}')


Saved: physics_results_with_usage.csv

Usage totals:
input_tokens      321469.0
output_tokens    1887098.0
total_tokens     2208567.0
cost              11.99071
dtype: Float64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>